# 1x1-conv-channel-reshape — ex2: 1×1 conv on rectangular (H≠W) input — output has same H, W and matches per-pixel Linear

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `1x1-conv-channel-reshape`. Running the final beacon cell reports progress against the `CNN: 1x1 conv channel-reshape` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1x1 conv channel-reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`1x1-conv-channel-reshape`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "1x1-conv-channel-reshape"
DD_SUBTOPIC = "CNN: 1x1 conv channel-reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 1×1 conv preserves H, W — even for rectangular inputs

Ex1 showed that a 1×1 `Conv2d` is per-pixel linear: it remixes the channel axis but leaves the spatial axes UNTOUCHED. The deepening move verifies this on a rectangular `H ≠ W` input — both axes pass through unchanged.

```
Conv2d(C_in, C_out, kernel_size=1, stride=1, padding=0)
input:  (B, C_in,  H, W)
output: (B, C_out, H, W)     # H, W identical to input
```

**Equivalence to per-pixel `nn.Linear`.** Flatten the spatial axes to `(B·H·W, C_in)`, apply `Linear(C_in, C_out)` with the conv's weight + bias reshaped, then un-flatten back to `(B, C_out, H, W)`. The two outputs match to machine precision.

**Why rectangular inputs are the load-bearing test.** A 1×1 conv on a square input could pass a misimplemented 'transpose H/W' bug accidentally. A non-square `(H=7, W=11)` input would FAIL if the implementation mixed H and W up. ARENA's vision pipelines see rectangular feature maps after asymmetric pooling — this is not a toy.

### Exercise 2 — 1×1 conv on rectangular (H≠W) input — output has same H, W and matches per-pixel Linear

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-pixel-Linear identity for a 1×1 Conv2d on a RECTANGULAR `(H ≠ W)` feature map: verify the output shape preserves both spatial axes AND matches an explicit `Linear(C_in, C_out)` computed pixel-by-pixel.
> Keywords: conv2d, 1x1, rectangular, per-pixel-linear
> ```

**KCs targeted:** `1x1-conv-preserves-spatial-axes`, `conv-weight-reshape-to-linear-weight`

Implement `ex2_rect_one_by_one_via_linear(x, conv)`. Same idea as ex1, but the input has DISTINCT spatial dims `(H ≠ W)` — for instance `(B=2, C_in=3, H=7, W=11)`.

Steps:
1. Read `conv.weight` (shape `(C_out, C_in, 1, 1)`) and `conv.bias` (shape `(C_out,)` or `None`).
2. Reshape the conv weight to a Linear weight of shape `(C_out, C_in)` — squeeze the trailing 1×1.
3. Use `einops.rearrange` to fold `H, W` into a single batch-like axis: `'b c h w -> (b h w) c'`.
4. Apply the Linear: `out_flat = x_flat @ W.T + bias` (broadcast bias correctly; if `conv.bias is None`, skip it).
5. `rearrange` back to `(b, c_out, h, w)`.

Inputs:
- `x`: shape `(B, C_in, H, W)` with possibly `H ≠ W`.
- `conv`: a `nn.Conv2d(C_in, C_out, kernel_size=1)`.

Output: a `Tensor` of shape `(B, C_out, H, W)` — identical to `conv(x)` to within fp32 atol.

In [ ]:
import einops as _einops

def ex2_rect_one_by_one_via_linear(x, conv):
    W = conv.weight                                  # (C_out, C_in, 1, 1)
    W_lin = W.squeeze(-1).squeeze(-1)                # (C_out, C_in)
    b, c_in, h, w = x.shape
    x_flat = _einops.rearrange(x, 'b c h w -> (b h w) c')  # ((b·h·w), C_in)
    out_flat = x_flat @ W_lin.t()                    # ((b·h·w), C_out)
    if conv.bias is not None:
        out_flat = out_flat + conv.bias
    return _einops.rearrange(
        out_flat, '(b h w) c -> b c h w', b=b, h=h, w=w
    )


<details><summary>Solution</summary>

```python
import einops as _einops

def ex2_rect_one_by_one_via_linear(x, conv):
    W = conv.weight                                  # (C_out, C_in, 1, 1)
    W_lin = W.squeeze(-1).squeeze(-1)                # (C_out, C_in)
    b, c_in, h, w = x.shape
    x_flat = _einops.rearrange(x, 'b c h w -> (b h w) c')  # ((b·h·w), C_in)
    out_flat = x_flat @ W_lin.t()                    # ((b·h·w), C_out)
    if conv.bias is not None:
        out_flat = out_flat + conv.bias
    return _einops.rearrange(
        out_flat, '(b h w) c -> b c h w', b=b, h=h, w=w
    )
```

**`squeeze(-1).squeeze(-1)` over `.reshape(C_out, C_in)`.** Either works for a 1×1 conv. `squeeze` makes the no-spatial intent explicit; `reshape` would silently work for any kernel size, potentially masking a bug if the conv weren't actually 1×1.

**`b=b, h=h, w=w` in the un-rearrange is load-bearing.** Without passing the axis sizes back, einops can't uniquely decompose the flat axis when `b·h·w` factors ambiguously (e.g. 2·7·11 = 154, but also 11·14 or 7·22). Explicit sizes pin the correct factorisation.

**Why rectangular catches the transpose bug.** A naive implementation that reshapes via `'b c h w -> (b w h) c'` (note: w before h) and reverses with `'(b h w) c'` will silently swap H ↔ W. On a square input the swap is invisible — the test would pass. Rectangular inputs make this bug an immediate shape mismatch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()